# 02_fasta2a_server

Stage 2 of the `codes/a2a/` track: **FastA2A Worker** — framework owns the
JSON-RPC surface; you implement `run_task`.

Notebook path: `codes/a2a/02_fasta2a_server/qa.ipynb`. Deps come from the shared
`codes/requirements.txt` (`fasta2a`, `uvicorn`, `httpx`). Stage 1
(`01_agentcard_sendmessage`) stays untouched as the stdlib wire lab.

Architecture to keep in your head:

`GET /.well-known/agent-card.json` → `POST /` `message/send` → Worker runs in the
background → poll `tasks/get` until `input-required` / `completed` → follow-up
with the same `context_id` (new task id).

Contrast with stage 1: PascalCase `SendMessage`, `TASK_STATE_*`, `ROLE_*`,
`/a2a/jsonrpc`, and continuing the **same** `taskId` are the 1.0.1 wire names.
FastA2A currently speaks `message/send`, kebab-case states, `user`/`agent`, and
always mints a new task per send.


In [1]:
from pathlib import Path
import sys
import subprocess
import time

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "02_fasta2a_server" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "a2a" / "02_fasta2a_server"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/a2a/02_fasta2a_server")

ROOT = lab_root()
sys.path.insert(0, str(ROOT))
print(ROOT)

from server import DEFAULT_HOST, DEFAULT_PORT, build_app
from client import wait_for_server, new_user_message, poll_task, _task_from_send, DEFAULT_BASE
import asyncio
import httpx
from fasta2a.client import A2AClient
from fasta2a.schema import Part

BASE = f"http://{DEFAULT_HOST}:{DEFAULT_PORT}"
_server_proc = subprocess.Popen(
    [sys.executable, str(ROOT / "server.py"), "--host", DEFAULT_HOST, "--port", str(DEFAULT_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
# block until card is up
card0 = await wait_for_server(BASE)
print("server ready →", BASE, "name=", card0.get("name"))


/workspace/AEFS-Notes/codes/a2a/02_fasta2a_server


server ready → http://127.0.0.1:41243 name= writer-agent


## 1. Read the Agent Card shape

Fetch `/.well-known/agent-card.json`. Print the agent `name`, every skill `id`,
and each `supportedInterfaces` entry as `protocolBinding protocolVersion @ url`.


In [2]:
# answer
import httpx
card = httpx.get(f"{BASE}/.well-known/agent-card.json").json()
print("name →", card["name"])
print("skills →", [s["id"] for s in card["skills"]])
for iface in card["supportedInterfaces"]:
    print(
        f"interface → {iface['protocolBinding']} "
        f"{iface['protocolVersion']} @ {iface['url']}"
    )


name → writer-agent
skills → ['draft_summary']
interface → JSONRPC 1.0 @ http://127.0.0.1:41243


## 2. message/send without targetLength → input-required

Send a user message with only a text part. FastA2A returns the task in
`submitted` immediately — poll `tasks/get` until `input-required`. Print
task id, context id, state, and the agent's question text from history.


In [3]:
# answer
async def step2():
    async with httpx.AsyncClient(timeout=10.0) as http:
        client = A2AClient(agent=BASE, http_client=http)
        first = new_user_message([Part(text="Summarize MCP vs A2A for a lab notebook.")])
        resp = await client.send_message(first)
        task = _task_from_send(resp)
        task = await poll_task(client, task["id"])
        print("task id    →", task["id"])
        print("context id →", task["context_id"])
        print("state      →", task["status"]["state"])
        ask = next(
            (p.get("text") for m in reversed(task["history"]) if m.get("role") == "agent"
             for p in m.get("parts") or [] if p.get("text")),
            None,
        )
        print("agent asks →", ask)
        return task["id"], task["context_id"]

TASK_ID, CONTEXT_ID = await step2()


task id    → 12af03e2-282f-489f-ab65-f7daf70d70dd
context id → 6ac48ba8-c375-4eb8-a751-397766784e34
state      → input-required
agent asks → Please send a data part with targetLength (e.g. '3 paragraphs').


## 3. Follow-up on the same context_id → completed + artifact

Send a second `message/send` with the same `context_id` and a data part
`{"targetLength": "3 paragraphs"}`. Expect a **new** task id. Poll until
`completed` and print the artifact text.


In [4]:
# answer
async def step3():
    async with httpx.AsyncClient(timeout=10.0) as http:
        client = A2AClient(agent=BASE, http_client=http)
        follow = new_user_message(
            [Part(data={"targetLength": "3 paragraphs"}, media_type="application/json")],
            context_id=CONTEXT_ID,
        )
        resp = await client.send_message(follow)
        task = _task_from_send(resp)
        print("new task id →", task["id"], "(same as first?", task["id"] == TASK_ID, ")")
        task = await poll_task(client, task["id"])
        print("state       →", task["status"]["state"])
        art = task["artifacts"][0]
        print("artifact    →", art["name"])
        print("content     →", art["parts"][0]["text"])

await step3()


new task id → 9041c79f-643a-4870-863b-1c6f050ad87f (same as first? False )
state       → completed
artifact    → summary
content     → [fasta2a-worker] length=3 paragraphs
source=Summarize MCP vs A2A for a lab notebook.
Done.


## 4. Stage-1 wire vs FastA2A names — write it once

In a few lines (print a string), contrast stage-1 stdlib wire names with what
FastA2A speaks: method names, roles, states, URL, and how follow-up works.


In [5]:
# answer
print(
    "Stage-1 stdlib: SendMessage / GetTask, ROLE_USER|ROLE_AGENT, "
    "TASK_STATE_*, POST /a2a/jsonrpc, A2A-Version: 1.0, continue same taskId. "
    "FastA2A: message/send / tasks/get, user|agent, kebab-case states, "
    "POST /, new task per send + context_id for conversation continuity."
)


Stage-1 stdlib: SendMessage / GetTask, ROLE_USER|ROLE_AGENT, TASK_STATE_*, POST /a2a/jsonrpc, A2A-Version: 1.0, continue same taskId. FastA2A: message/send / tasks/get, user|agent, kebab-case states, POST /, new task per send + context_id for conversation continuity.


## Cleanup

Stop the background server when you are done exploring.


In [6]:
_server_proc.terminate()
try:
    _server_proc.wait(timeout=5)
except Exception:
    _server_proc.kill()
print("server stopped")


server stopped
